# Classification Tasks - Indonesian (PRDECT-ID) Dataset

In [ ]:
# config

!pip install PySastrawi -q

In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

from sklearn.svm import SVC
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

**1. Load Dataset**

In [ ]:
df = pd.read_csv('...', sep=',') # File path dataset
df = df[['Customer Review', 'Sentiment']]
df.columns = ['review', 'sentiment']
df.head()

**2. Check class distribution & missing values**

In [ ]:
print(df['sentiment'].value_counts())
print('\nMissing values: ')
print(df.isnull().sum())

**3. Text Cleaning**

In [ ]:
def clean_text(txt):
    txt = str(txt)
    txt = txt.lower()
    txt = re.sub(r'http\S+', '', txt)
    txt = re.sub(r'[^a-zA-Z\s]', '', txt)
    txt = re.sub(r'\s+', ' ', txt).strip()
    return txt

df['review_clean'] = df['review'].apply(clean_text)
df[['review', 'review_clean']].head()

**4. Stopword Removal + Stemming**

In [ ]:
factory = StemmerFactory()
stemmer = factory.create_stemmer()

stop_factory = StopWordRemoverFactory()
stopwords = stop_factory.get_stop_words()

def preprocess(txt):
  tokens = txt.split()
  tokens = [w for w in tokens if w not in stopwords]
  tokens = [stemmer.stem(w) for w in tokens]
  return ' '.join(tokens)

df['review_preprocessed'] = df['review_clean'].apply(preprocess)
df[['review_clean', 'review_preprocessed']].head()

**5. TF-IDF + Split Data**

In [ ]:
tfidf = TfidfVectorizer(max_features=3000)

X = tfidf.fit_transform(df['review_preprocessed'])
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Total data : {X.shape[0]}')
print(f'Jumlah fitur TF-IDF : {X.shape[1]}')
print(f'Data training : {X_train.shape[0]}')
print(f'Data testing  : {X_test.shape[0]}')

**6. Training SVC**

In [ ]:
svc = SVC(kernel='rbf', C=10, gamma=1, random_state=42)
svc.fit(X_train, y_train)

print('Training selesai.')
print(f'Jumlah support vector : {sum(svc.n_support_)}')

**7. Evaluation**

In [ ]:
y_pred = svc.predict(X_test)

print('Accuracy:', accuracy_score(y_test, y_pred))
print()
print('Classification Report:')
print(classification_report(y_test, y_pred))

**8. Confusion Matrix**

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=['Negative', 'Positive'])

plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negative', 'Positive'],
            yticklabels=['Negative', 'Positive'])

plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - SVC PRDECT-ID')
plt.tight_layout()
plt.show()

**9. Save model**

In [ ]:
joblib.dump(svc, 'svc_model.pkl')
joblib.dump(tfidf, 'tfidf_vectorizer.pkl')
print('Model saved!')